# Claude 基础用法 — Anthropic SDK

本 Notebook 演示如何通过 **七牛 AIToken** 平台，使用 **Anthropic Python SDK** 直接调用 Claude Messages API 进行文本生成。

## 功能简介

Anthropic Python SDK 是调用 Claude 模型最直接的方式，无需 Agent 框架即可完成各类文本生成任务：
- **非流式生成**：一次性返回完整响应
- **流式生成**：逐步返回响应内容，降低首字延迟
- **系统提示词**：自定义模型角色和行为
- **多轮对话**：手动管理消息历史，实现上下文连续对话

**API 端点**：`https://api.qnaigc.com`  
**适用模型**：`claude-4.6-sonnet`

## 1. 环境准备

安装 `anthropic` 依赖包，并设置 API Key。

In [ ]:
# 安装 Anthropic Python SDK
%pip install anthropic -q

In [ ]:
import os
from anthropic import Anthropic

# 七牛 AIToken 平台地址
BASE_URL = "https://api.qnaigc.com"

# 从环境变量读取 API Key（或替换为你的 API Key）
API_KEY = os.environ.get("QINIU_API_KEY", "<your-api-key>")

# 使用的模型
MODEL = "claude-4.6-sonnet"

# 初始化 Anthropic 客户端，指向七牛 AIToken 平台
client = Anthropic(
    api_key=API_KEY,
    base_url=BASE_URL,
)

print("环境配置完成!")
print(f"  API 端点: {BASE_URL}")
print(f"  模型: {MODEL}")

## 2. 非流式生成

使用 `client.messages.create()` 进行最基础的文本生成调用，一次性返回完整响应。

In [ ]:
# 基础文本生成
message = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "请用一句话介绍什么是大语言模型。"}
    ],
)

# 打印模型回复
print("=== 模型回复 ===")
print(message.content[0].text)

# 打印 Token 用量信息
print(f"\n--- 用量信息 ---")
print(f"模型: {message.model}")
print(f"停止原因: {message.stop_reason}")
print(f"输入 Tokens: {message.usage.input_tokens}")
print(f"输出 Tokens: {message.usage.output_tokens}")

## 3. 流式生成

使用 `stream=True` 参数开启流式响应，逐步接收生成内容。适用于需要降低首字延迟、实时展示输出的场景。

In [ ]:
# 流式生成
print("=== 流式输出 ===")
with client.messages.stream(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "请简要介绍 Python 语言的三个核心优势。"}
    ],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

# 获取最终的完整消息对象
response = stream.get_final_message()
print(f"\n\n--- 用量信息 ---")
print(f"输入 Tokens: {response.usage.input_tokens}")
print(f"输出 Tokens: {response.usage.output_tokens}")

## 4. 系统提示词

通过 `system` 参数设置系统提示词，定义模型的角色和行为方式。

In [ ]:
# 使用系统提示词自定义模型行为
message = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system="你是一位资深的 Python 开发工程师，擅长用简洁清晰的方式解释技术概念。回答时请附上代码示例。",
    messages=[
        {"role": "user", "content": "什么是 Python 装饰器？"}
    ],
)

print("=== 模型回复 ===")
print(message.content[0].text)

print(f"\n--- 用量信息 ---")
print(f"输入 Tokens: {message.usage.input_tokens}")
print(f"输出 Tokens: {message.usage.output_tokens}")

## 5. 多轮对话

通过手动维护 `messages` 列表，将历史对话作为上下文传入，实现多轮连续对话。

消息角色说明：
- `user`：用户消息
- `assistant`：模型回复

In [ ]:
# 多轮对话：手动维护消息历史
conversation = []

def chat(user_message: str) -> str:
    """发送消息并获取回复，自动维护对话历史。"""
    conversation.append({"role": "user", "content": user_message})

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system="你是一位旅行顾问，擅长推荐旅行目的地和规划行程。",
        messages=conversation,
    )

    assistant_message = response.content[0].text
    # 将模型回复也加入对话历史，保持上下文
    conversation.append({"role": "assistant", "content": assistant_message})

    return assistant_message


# 第一轮对话
print("=== 第一轮 ===")
print(f"用户: 我想在五月份去一个适合拍照的地方旅行，有什么推荐？")
reply = chat("我想在五月份去一个适合拍照的地方旅行，有什么推荐？")
print(f"Claude: {reply}")

# 第二轮对话（模型会记住上一轮的推荐内容）
print(f"\n=== 第二轮 ===")
print(f"用户: 你推荐的第一个地方，帮我规划一个 3 天的行程吧。")
reply = chat("你推荐的第一个地方，帮我规划一个 3 天的行程吧。")
print(f"Claude: {reply}")

print(f"\n--- 对话历史长度: {len(conversation)} 条消息 ---")

## 6. 生成参数调整

通过 `temperature`、`top_p` 等参数控制生成行为。

In [ ]:
# 低温度：更确定性的输出，适合事实性问答
message_low_temp = client.messages.create(
    model=MODEL,
    max_tokens=256,
    temperature=0.0,
    messages=[
        {"role": "user", "content": "Python 中 list 和 tuple 的区别是什么？请用一句话总结。"}
    ],
)

print("=== 低温度 (temperature=0.0) ===")
print(message_low_temp.content[0].text)

# 高温度：更多样化的输出，适合创意写作
message_high_temp = client.messages.create(
    model=MODEL,
    max_tokens=256,
    temperature=1.0,
    messages=[
        {"role": "user", "content": "请用一个比喻来描述编程。"}
    ],
)

print(f"\n=== 高温度 (temperature=1.0) ===")
print(message_high_temp.content[0].text)

## 7. 参数说明

### Anthropic 客户端初始化

| 参数 | 类型 | 说明 |
|------|------|------|
| `api_key` | string | API Key，使用七牛 AIToken 的 API Key |
| `base_url` | string | API 端点地址，设为 `https://api.qnaigc.com` 指向七牛 AIToken 平台 |

### messages.create() 主要参数

| 参数 | 类型 | 必填 | 说明 |
|------|------|------|------|
| `model` | string | 是 | 模型 ID，如 `claude-4.6-sonnet` |
| `max_tokens` | integer | 是 | 最大输出 Token 数 |
| `messages` | array | 是 | 消息列表，包含 `role`（`user`/`assistant`）和 `content` |
| `system` | string | 否 | 系统提示词，定义模型角色和行为 |
| `temperature` | float | 否 | 采样温度，范围 0~1，默认 1。越低越确定性，越高越多样化 |
| `top_p` | float | 否 | 核采样参数，范围 0~1，与 temperature 二选一使用 |
| `top_k` | integer | 否 | 每步采样的候选 Token 数量 |
| `stop_sequences` | array | 否 | 自定义停止序列，模型遇到这些字符串时停止生成 |

### 响应对象

| 字段 | 说明 |
|------|------|
| `message.content[0].text` | 模型的文本回复 |
| `message.model` | 实际使用的模型 ID |
| `message.stop_reason` | 停止原因（`end_turn`：正常结束，`max_tokens`：达到长度限制） |
| `message.usage.input_tokens` | 输入 Token 数 |
| `message.usage.output_tokens` | 输出 Token 数 |

## 8. 与 Claude Agent SDK 的对比

| 特性 | Anthropic SDK（本示例） | Claude Agent SDK |
|------|------------------------|------------------|
| 调用方式 | `client.messages.create()` 同步调用 | `query()` 异步迭代器 |
| 多轮对话 | 手动维护 `messages` 列表 | `ClaudeSDKClient` 自动管理 |
| 工具调用 | 需手动定义 `tools` 参数并处理回调 | `@tool` 装饰器 + MCP Server 自动管理 |
| 流式响应 | `client.messages.stream()` | 内置流式消息 |
| 依赖包 | `anthropic` | `claude-agent-sdk` |
| 适用场景 | 轻量级、精确控制 API 调用 | 需要工具调用、多轮 Agent 交互 |

## 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [Anthropic Python SDK 文档](https://github.com/anthropics/anthropic-sdk-python)
- [Anthropic Messages API 参考](https://docs.anthropic.com/en/api/messages)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)